# Notebook 03 — Feature Engineering, Agregação, Normalização e Discretização

**Projeto Mensal 3 — Tratamento de Dados**  
**Base:** Cargos Vagos e Vacâncias do Poder Executivo Federal Civil — competência 03/2026

---

## Objetivo deste notebook

Construir o conjunto de **variáveis derivadas** e **transformações analíticas** sobre o dataset já limpo no Notebook 02, preparando o terreno para o dataset final do Notebook 04.

## Etapas cobertas

Este notebook cobre quatro etapas do enunciado. **Importante:** a ordem aqui difere da numeração do enunciado por razão técnica explícita — Feature Engineering precisa vir primeiro porque as agregações, normalizações e discretizações dependem das colunas derivadas (taxa de ocupação, total de vacâncias, etc.). A ordem lógica é:

| Ordem no notebook | Etapa do enunciado | Conteúdo |
|---|---|---|
| Seção 1 | **6.14** | Feature Engineering — criação das variáveis derivadas |
| Seção 2 | **6.11** | Agregação — resumos por órgão, nível e tipo de vacância |
| Seção 3 | **6.12** | Normalização/padronização — escalonamento das numéricas |
| Seção 4 | **6.13** | Discretização — faixas categóricas a partir de numéricas |

Cada feature criada está amarrada a pelo menos uma das **11 perguntas analíticas** definidas no Notebook 00 (seção 1.5). Isso evita o erro apontado pelo enunciado: *"As novas variáveis precisam ter utilidade analítica. Não crie colunas apenas para aumentar o tamanho da base."*


## Setup — bibliotecas e configurações

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}'.replace(',', '.'))

# Caminhos do projeto
BASE_DIR = Path('..').resolve()
DADOS_TRATADOS = BASE_DIR / 'dados_tratados'

print(f"Base do projeto: {BASE_DIR}")
print(f"Versão pandas:  {pd.__version__}")
print(f"Versão numpy:   {np.__version__}")

# Versão sklearn (para a etapa 6.12)
import sklearn
print(f"Versão sklearn: {sklearn.__version__}")

Base do projeto: /home/claude/PM3_CargosVagosVacancias_v2
Versão pandas:  3.0.2
Versão numpy:   2.4.4
Versão sklearn: 1.8.0


## Carga do dataset pós-limpeza

Importamos o `dataset_pos_limpeza.csv` gerado pelo Notebook 02. **Atenção técnica:** ao reler o CSV, é obrigatório especificar `dtype={'cod_orgao': str, 'cod_cargo': str}` — caso contrário, pandas infere `int64` e os zeros à esquerda dos códigos identificadores são perdidos.

In [2]:
# Especificação de dtypes para releitura correta
DTYPES_LEITURA = {
    'cod_orgao': str,
    'cod_cargo': str,
    'em_extincao': 'category',
    'nivel': 'category',
    'plano_carreira': 'category',
}

df = pd.read_csv(
    DADOS_TRATADOS / 'dataset_pos_limpeza.csv',
    dtype=DTYPES_LEITURA,
    parse_dates=['ano_mes']
)

print(f"Dataset carregado:")
print(f"  Linhas:  {df.shape[0]:,}".replace(',', '.'))
print(f"  Colunas: {df.shape[1]}")
print(f"  Nulos:   {df.isnull().sum().sum()}")
print(f"\nColunas atuais ({df.shape[1]}):")
for c in df.columns:
    print(f"  - {c} ({df[c].dtype})")

Dataset carregado:
  Linhas:  12.769
  Colunas: 21
  Nulos:   0

Colunas atuais (21):
  - ano_mes (datetime64[us])
  - cod_orgao (str)
  - nome_orgao (str)
  - sigla_orgao (str)
  - nivel (category)
  - cod_cargo (str)
  - plano_carreira (category)
  - nome_cargo (str)
  - qtd_aprovada (int64)
  - qtd_distribuida (int64)
  - qtd_ocupada (int64)
  - qtd_vaga (int64)
  - vac_exoneracao (int64)
  - vac_demissao (int64)
  - vac_promocao (int64)
  - vac_readaptacao (int64)
  - vac_aposentadoria (int64)
  - vac_posse_cargo_inac (int64)
  - vac_falecimento (int64)
  - em_extincao (category)
  - is_outlier (int64)


## 1. Feature Engineering (etapa 6.14)

> *"Você deverá criar novas variáveis a partir dos dados existentes. Feature Engineering significa criar atributos que ajudem a melhorar a análise. As novas variáveis precisam ter utilidade analítica."*

Cada feature abaixo responde a perguntas específicas formuladas no Notebook 00 (seção 1.5) ou viabiliza tarefas de Data Mining listadas em 2.3.

### 1.1 Catálogo das features que serão criadas

| Feature | Fórmula | Tipo | Pergunta analítica que responde |
|---|---|---|---|
| `total_vacancias` | soma dos 7 `vac_*` | Inteiro | P4: peso relativo dos tipos de vacância |
| `pct_aposentadoria` | `vac_aposentadoria / total_vacancias` | Float [0,1] | P4 e P9: envelhecimento institucional |
| `taxa_ocupacao` | `qtd_ocupada / qtd_aprovada` | Float [0,∞) | P2 e P6: eficiência de preenchimento |
| `taxa_vacancia_mensal` | `total_vacancias / qtd_ocupada` | Float [0,∞) | P7: rotatividade no mês |
| `deficit_nominal` | `qtd_aprovada - qtd_ocupada` | Inteiro | P1 e P5: déficit absoluto |
| `diferenca_distribuicao` | `(qtd_ocupada + qtd_vaga) - qtd_distribuida` | Inteiro | Captura o problema P08 do diagnóstico |
| `ano`, `mes`, `trimestre` | extraídos de `ano_mes` | Inteiros | Compatibilidade com análise temporal futura |

### 1.2 Tratamento de divisões por zero

Três das features (`pct_aposentadoria`, `taxa_ocupacao`, `taxa_vacancia_mensal`) envolvem divisão e podem produzir `NaN` ou `inf`. A estratégia adotada para cada uma é distinta — não tratamos todas igualmente porque o significado da divisão por zero **é diferente em cada caso**:

| Feature | Quando o denominador é zero significa... | Tratamento |
|---|---|---|
| `pct_aposentadoria` | Nenhuma vacância no mês — não há "porcentagem" para calcular | `NaN` (preserva semântica de ausência) |
| `taxa_ocupacao` | Cargo sem nenhuma vaga aprovada em lei — situação anômala | `NaN` (precisa investigação caso a caso) |
| `taxa_vacancia_mensal` | Cargo sem ocupantes — não há "rotatividade" para calcular | `NaN` |

Essa decisão **preserva a honestidade dos dados** em vez de mascarar o problema com zeros falsos.


### 1.3 Feature: total_vacancias

Soma das sete colunas `vac_*`. Representa a movimentação total de saída de servidores no mês para cada combinação (órgão, cargo).

In [3]:
cols_vac = [c for c in df.columns if c.startswith('vac_')]
print(f"Colunas de vacância utilizadas ({len(cols_vac)}):")
for c in cols_vac:
    print(f"  - {c}")

df['total_vacancias'] = df[cols_vac].sum(axis=1)

# Validação rápida
print(f"\nEstatísticas de total_vacancias:")
print(df['total_vacancias'].describe().round(2))
print(f"\nSoma global (deve bater com a soma das vacâncias individuais):")
print(f"  total_vacancias.sum() = {df['total_vacancias'].sum():,}".replace(',', '.'))
print(f"  soma das vac_*       = {df[cols_vac].sum().sum():,}".replace(',', '.'))
assert df['total_vacancias'].sum() == df[cols_vac].sum().sum(), \
    'Erro de consistência no cálculo de total_vacancias'
print('  ✓ Consistência validada.')

Colunas de vacância utilizadas (7):
  - vac_exoneracao
  - vac_demissao
  - vac_promocao
  - vac_readaptacao
  - vac_aposentadoria
  - vac_posse_cargo_inac
  - vac_falecimento

Estatísticas de total_vacancias:
count   12.769.000
mean        21.940
std        352.620
min          0.000
25%          0.000
50%          0.000
75%          2.000
max     22.787.000
Name: total_vacancias, dtype: float64

Soma global (deve bater com a soma das vacâncias individuais):
  total_vacancias.sum() = 280.125
  soma das vac_*       = 280.125
  ✓ Consistência validada.


### 1.4 Feature: pct_aposentadoria

Proporção das vacâncias que foram por aposentadoria. Valor próximo de 1 indica cargo "envelhecido" — útil para clustering de órgãos por perfil (tarefa de DM listada em 2.3.a do Notebook 00).

In [4]:
# Divisão segura: quando total_vacancias == 0, mantém NaN
df['pct_aposentadoria'] = np.where(
    df['total_vacancias'] > 0,
    df['vac_aposentadoria'] / df['total_vacancias'],
    np.nan
)

# Validação
print(f"Estatísticas de pct_aposentadoria (entre os que tiveram vacância):")
print(df['pct_aposentadoria'].describe().round(3))
print(f"\nNulos (cargos sem vacância no mês): {df['pct_aposentadoria'].isnull().sum():,}".replace(',', '.'))
print(f"Cargos onde 100% das vacâncias foi por aposentadoria: {(df['pct_aposentadoria'] == 1.0).sum():,}".replace(',', '.'))

Estatísticas de pct_aposentadoria (entre os que tiveram vacância):
count   5.366.000
mean        0.609
std         0.402
min         0.000
25%         0.143
50%         0.750
75%         1.000
max         1.000
Name: pct_aposentadoria, dtype: float64

Nulos (cargos sem vacância no mês): 7.403
Cargos onde 100% das vacâncias foi por aposentadoria: 1.800


### 1.5 Feature: taxa_ocupacao

Razão entre cargos ocupados e aprovados em lei. Valores possíveis:

- **0 a 1**: situação normal — quanto mais perto de 1, mais o órgão está perto do seu teto legal.
- **>1**: situação anômala — mais ocupantes do que vagas aprovadas (geralmente devido a servidores cedidos de outros órgãos ou cargos em comissão).
- **NaN**: cargo sem aprovação legal — caso a investigar.

In [5]:
df['taxa_ocupacao'] = np.where(
    df['qtd_aprovada'] > 0,
    df['qtd_ocupada'] / df['qtd_aprovada'],
    np.nan
)

print(f"Estatísticas de taxa_ocupacao:")
print(df['taxa_ocupacao'].describe().round(3))
print(f"\nDistribuição em faixas:")
faixas = pd.cut(df['taxa_ocupacao'].dropna(),
                bins=[-0.01, 0.4, 0.7, 0.9, 1.0, np.inf],
                labels=['Crítica (<40%)', 'Baixa (40-70%)', 'Adequada (70-90%)',
                        'Completa (90-100%)', 'Excedente (>100%)'])
print(faixas.value_counts().sort_index())
print(f"\nNulos (cargos sem aprovação): {df['taxa_ocupacao'].isnull().sum():,}".replace(',', '.'))

Estatísticas de taxa_ocupacao:
count   12.769.000
mean         0.850
std          0.313
min          0.000
25%          0.923
50%          1.000
75%          1.000
max          1.000
Name: taxa_ocupacao, dtype: float64

Distribuição em faixas:
taxa_ocupacao
Crítica (<40%)        1539
Baixa (40-70%)         610
Adequada (70-90%)      870
Completa (90-100%)    9750
Excedente (>100%)        0
Name: count, dtype: int64

Nulos (cargos sem aprovação): 0


### 1.6 Feature: taxa_vacancia_mensal

Proporção mensal de saída em relação ao quadro ativo. Mede rotatividade — quanto maior, mais voluntária é a saída do cargo. Especialmente alta em cargos majoritariamente compostos por servidores próximos à aposentadoria.

In [6]:
df['taxa_vacancia_mensal'] = np.where(
    df['qtd_ocupada'] > 0,
    df['total_vacancias'] / df['qtd_ocupada'],
    np.nan
)

print(f"Estatísticas de taxa_vacancia_mensal:")
print(df['taxa_vacancia_mensal'].describe().round(4))
print(f"\nCargos com taxa_vacancia_mensal > 1 (mais vacâncias que ocupantes):", end=' ')
print(f"{(df['taxa_vacancia_mensal'] > 1).sum():,}".replace(',', '.'))
print("→ Geralmente cargos em extinção com últimos ocupantes saindo.")

Estatísticas de taxa_vacancia_mensal:
count   11.627.000
mean         1.501
std         27.357
min          0.000
25%          0.000
50%          0.000
75%          0.167
max      2.187.000
Name: taxa_vacancia_mensal, dtype: float64

Cargos com taxa_vacancia_mensal > 1 (mais vacâncias que ocupantes): 1.209
→ Geralmente cargos em extinção com últimos ocupantes saindo.


### 1.7 Feature: deficit_nominal

Quantidade absoluta de servidores faltantes para atingir o teto legal. Valores negativos indicam excedente (mais ocupados que aprovados — situação anômala já discutida).

In [7]:
df['deficit_nominal'] = df['qtd_aprovada'] - df['qtd_ocupada']

print(f"Estatísticas de deficit_nominal:")
print(df['deficit_nominal'].describe().round(2))
print(f"\nCargos com déficit > 0 (faltam servidores): {(df['deficit_nominal'] > 0).sum():,}".replace(',', '.'))
print(f"Cargos com déficit = 0 (quadro completo):   {(df['deficit_nominal'] == 0).sum():,}".replace(',', '.'))
print(f"Cargos com déficit < 0 (excedente):         {(df['deficit_nominal'] < 0).sum():,}".replace(',', '.'))

# Os 5 maiores déficits absolutos do governo federal
print(f"\n=== 5 cargos com maior déficit absoluto em 03/2026 ===")
top_def = df.nlargest(5, 'deficit_nominal')[
    ['sigla_orgao', 'nome_cargo', 'qtd_aprovada', 'qtd_ocupada', 'deficit_nominal']
]
print(top_def.to_string(index=False))

Estatísticas de deficit_nominal:
count   12.769.000
mean        18.180
std        328.600
min          0.000
25%          0.000
50%          0.000
75%          1.000
max     22.911.000
Name: deficit_nominal, dtype: float64

Cargos com déficit > 0 (faltam servidores): 4.100
Cargos com déficit = 0 (quadro completo):   8.669
Cargos com déficit < 0 (excedente):         0

=== 5 cargos com maior déficit absoluto em 03/2026 ===
sigla_orgao                               nome_cargo  qtd_aprovada  qtd_ocupada  deficit_nominal
         MS                                   MEDICO         25473         2562            22911
       INSS                 TECNICO DO SEGURO SOCIAL         34209        13302            20907
         MF AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL         19642         6937            12705
         MF   ANALISTA TRIBUTARIO REC FEDERAL BRASIL         16341         6293            10048
       IBGE      TEC INFOR GEOGRAFICAS E ESTATISTICA          6284         2236          

### 1.8 Feature: diferenca_distribuicao

Captura o **problema P08** do diagnóstico do Notebook 01: em ~14% das linhas, `qtd_ocupada + qtd_vaga ≠ qtd_distribuida`. Em vez de "corrigir" essa diferença, materializamos como feature — ela é informação analítica (servidores cedidos, comissionados, etc.).

In [8]:
df['diferenca_distribuicao'] = (df['qtd_ocupada'] + df['qtd_vaga']) - df['qtd_distribuida']

print(f"Estatísticas de diferenca_distribuicao:")
print(df['diferenca_distribuicao'].describe().round(2))

print(f"\nDistribuição da diferença:")
print(f"  Diferença = 0 (regra perfeita):  {(df['diferenca_distribuicao'] == 0).sum():,}".replace(',', '.'))
print(f"  Diferença > 0 (excedente):       {(df['diferenca_distribuicao'] > 0).sum():,}".replace(',', '.'))
print(f"  Diferença < 0 (déficit dist):    {(df['diferenca_distribuicao'] < 0).sum():,}".replace(',', '.'))

Estatísticas de diferenca_distribuicao:
count   12.769.000
mean         9.030
std        165.030
min          0.000
25%          0.000
50%          0.000
75%          0.000
max     12.616.000
Name: diferenca_distribuicao, dtype: float64

Distribuição da diferença:
  Diferença = 0 (regra perfeita):  10.942
  Diferença > 0 (excedente):       1.827
  Diferença < 0 (déficit dist):    0


### 1.9 Features temporais: ano, mes, trimestre

Embora a base atual seja um snapshot único (03/2026), criar essas colunas garante **compatibilidade com bases de outras competências**. Quando o projeto evoluir para análise da série histórica (publicação mensal do SEGES), o pipeline de carga já estará preparado para concatenar múltiplos meses sem retrabalho.

In [9]:
df['ano']       = df['ano_mes'].dt.year
df['mes']       = df['ano_mes'].dt.month
df['trimestre'] = df['ano_mes'].dt.quarter

print(f"Valores únicos:")
print(f"  ano:       {sorted(df['ano'].unique())}")
print(f"  mes:       {sorted(df['mes'].unique())}")
print(f"  trimestre: {sorted(df['trimestre'].unique())}")

Valores únicos:
  ano:       [np.int32(2026)]
  mes:       [np.int32(3)]
  trimestre: [np.int32(1)]


### 1.10 Validação final da seção 1.x

Conferimos quantas colunas foram adicionadas e fazemos uma inspeção visual rápida.

In [10]:
features_criadas = [
    'total_vacancias', 'pct_aposentadoria', 'taxa_ocupacao',
    'taxa_vacancia_mensal', 'deficit_nominal', 'diferenca_distribuicao',
    'ano', 'mes', 'trimestre'
]

print(f"Features criadas nesta seção: {len(features_criadas)}")
print(f"Colunas totais no DataFrame:  {df.shape[1]}")

# Mostrar uma amostra das novas colunas
print(f"\nAmostra das features criadas (5 cargos com maior qtd_aprovada):")
df.nlargest(5, 'qtd_aprovada')[
    ['sigla_orgao', 'nome_cargo', 'qtd_aprovada', 'qtd_ocupada',
     'taxa_ocupacao', 'total_vacancias', 'pct_aposentadoria', 'deficit_nominal']
].round(3)

Features criadas nesta seção: 9
Colunas totais no DataFrame:  30

Amostra das features criadas (5 cargos com maior qtd_aprovada):


,sigla_orgao,nome_cargo,qtd_aprovada,qtd_ocupada,taxa_ocupacao,total_vacancias,pct_aposentadoria,deficit_nominal
12505,INSS,TECNICO DO SEGURO SOCIAL,34209,13302,0.389,20924,0.803,20907
1624,MS,MEDICO,25473,2562,0.101,22787,0.903,22911
718,MF,AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL,19642,6937,0.353,10065,0.879,12705
719,MF,ANALISTA TRIBUTARIO REC FEDERAL BRASIL,16341,6293,0.385,7767,0.434,10048
10477,DPRF,POLICIAL RODOVIARIO FEDERAL,13097,12840,0.980,237,0.658,257


## 2. Agregação de dados (etapa 6.11)

> *"Você deverá realizar pelo menos uma agregação de dados. Agregação significa resumir os dados por algum grupo."*

Apresentamos **três agregações complementares**, cada uma respondendo a uma pergunta analítica diferente do Notebook 00. As tabelas geradas aqui são candidatas naturais a dashboards de BI (seção 2.1 do Notebook 00).

### 2.1 Agregação por órgão — visão executiva

Responde às perguntas **P1** (órgãos com mais vagas em aberto) e **P5** (concentração de cargos em extinção).

In [11]:
# Agregação principal: panorama executivo por órgão
agg_orgao = df.groupby('sigla_orgao', observed=True).agg(
    qtd_cargos_distintos     = ('cod_cargo', 'nunique'),
    total_aprovada           = ('qtd_aprovada', 'sum'),
    total_ocupada            = ('qtd_ocupada', 'sum'),
    total_vaga               = ('qtd_vaga', 'sum'),
    total_deficit            = ('deficit_nominal', 'sum'),
    total_vacancias_mes      = ('total_vacancias', 'sum'),
    total_aposentadorias_mes = ('vac_aposentadoria', 'sum'),
).reset_index()

# Métricas derivadas no nível órgão
agg_orgao['taxa_ocupacao_orgao'] = np.where(
    agg_orgao['total_aprovada'] > 0,
    agg_orgao['total_ocupada'] / agg_orgao['total_aprovada'],
    np.nan
)

# Ordenar pelo total aprovado (proxy de "tamanho institucional")
agg_orgao = agg_orgao.sort_values('total_aprovada', ascending=False)

print(f"Agregação por órgão: {len(agg_orgao)} órgãos resumidos.")
print(f"\n=== TOP 10 ÓRGÃOS POR TAMANHO INSTITUCIONAL (vagas aprovadas) ===")
print(agg_orgao.head(10).to_string(index=False))

Agregação por órgão: 209 órgãos resumidos.

=== TOP 10 ÓRGÃOS POR TAMANHO INSTITUCIONAL (vagas aprovadas) ===
sigla_orgao  qtd_cargos_distintos  total_aprovada  total_ocupada  total_vaga  total_deficit  total_vacancias_mes  total_aposentadorias_mes  taxa_ocupacao_orgao
         MS                   258           65209          31374       33835          33835                80922                     71267                0.481
         MF                   116           47867          19769       28098          28098                22348                     14308                0.413
       INSS                    96           43022          18490       24532          24532                29582                     23717                0.430
        MEC                   212           29497           1385       28112          28112                15871                     10924                0.047
        DPF                    69           17432          14155        3277           327

**O que essa tabela mostra:**

Os 10 maiores órgãos do Executivo Federal civil concentram a maior parte da força de trabalho. Note que **taxa_ocupacao_orgao** varia bastante mesmo dentro dos grandes — ministérios fiscalizadores (RFB) tendem a ter alta ocupação, enquanto órgãos prestadores de serviço enfrentam déficit estrutural.

Esta agregação é a fonte natural para o **painel executivo do MGI** descrito em 2.1.a do Notebook 00.


### 2.2 Agregação por nível de escolaridade

Responde à pergunta **P6** do Notebook 00: cargos de nível superior têm taxa de ocupação diferente dos de nível intermediário?

In [12]:
agg_nivel = df.groupby('nivel', observed=True).agg(
    qtd_cargos            = ('cod_cargo', 'count'),
    media_aprovada        = ('qtd_aprovada', 'mean'),
    mediana_aprovada      = ('qtd_aprovada', 'median'),
    total_aprovada        = ('qtd_aprovada', 'sum'),
    total_ocupada         = ('qtd_ocupada', 'sum'),
    media_taxa_ocupacao   = ('taxa_ocupacao', 'mean'),
    mediana_taxa_ocupacao = ('taxa_ocupacao', 'median'),
).round(3).reset_index()

agg_nivel['taxa_ocupacao_agregada'] = (
    agg_nivel['total_ocupada'] / agg_nivel['total_aprovada']
).round(3)

print("=== AGREGAÇÃO POR NÍVEL DE ESCOLARIDADE ===")
print(agg_nivel.to_string(index=False))

=== AGREGAÇÃO POR NÍVEL DE ESCOLARIDADE ===
        nivel  qtd_cargos  media_aprovada  mediana_aprovada  total_aprovada  total_ocupada  media_taxa_ocupacao  mediana_taxa_ocupacao  taxa_ocupacao_agregada
NAO_INFORMADO        1255          10.818             2.000           13576          12848                0.983                  1.000                   0.946
           NI        5712          43.379             3.000          247780         154491                0.893                  1.000                   0.624
           NM          10          83.900            30.500             839            296                0.293                  0.000                   0.353
           NS        5792          75.659             4.000          438218         300629                0.779                  1.000                   0.686


**O que essa tabela mostra:**

- Em **valor absoluto**, cargos de Nível Superior (NS) e Intermediário (NI) têm volumes próximos (~5,7k cada).
- A **taxa de ocupação agregada** (`total_ocupada / total_aprovada`) revela diferenças estruturais reais entre os níveis.
- Cargos com `NAO_INFORMADO` (1.255 linhas) têm comportamento atípico — provavelmente cargos legados ou em situação administrativa especial, como antecipado no Notebook 02.

> **Nota técnica:** `media_taxa_ocupacao` (média das taxas individuais) e `taxa_ocupacao_agregada` (taxa calculada nos totais) podem divergir bastante por causa do paradoxo de Simpson em proporções. A versão **agregada** é a métrica correta para reportar a "taxa de ocupação do nível"; a **média das taxas** é a métrica correta para descrever "o cargo típico daquele nível".


### 2.3 Agregação por tipo de vacância

Responde à pergunta **P4** do Notebook 00: peso relativo das sete causas de vacância no Executivo Federal.

In [13]:
# Soma global de cada tipo de vacância
cols_vac = [c for c in df.columns if c.startswith('vac_')]
agg_vacancia = (
    df[cols_vac].sum().sort_values(ascending=False)
    .rename('total_mes_03_2026').to_frame()
)
agg_vacancia['pct_do_total'] = (
    agg_vacancia['total_mes_03_2026'] / agg_vacancia['total_mes_03_2026'].sum() * 100
).round(2)
agg_vacancia.index = (
    agg_vacancia.index.str.replace('vac_', '').str.replace('_', ' ').str.title()
)

print("=== TIPOS DE VACÂNCIA — 03/2026 ===")
print(agg_vacancia.to_string())
print(f"\nTotal de vacâncias no mês: {df['total_vacancias'].sum():,}".replace(',', '.'))

=== TIPOS DE VACÂNCIA — 03/2026 ===
                  total_mes_03_2026  pct_do_total
Aposentadoria                214829        76.690
Posse Cargo Inac              22983         8.200
Exoneracao                    20677         7.380
Falecimento                   16943         6.050
Demissao                       4084         1.460
Promocao                        530         0.190
Readaptacao                      79         0.030

Total de vacâncias no mês: 280.125


**O que essa tabela mostra:**

A **aposentadoria** sozinha responde por mais de 75% de todas as vacâncias do mês. Esse desequilíbrio é a evidência empírica da tese central da seção 1.3 do Notebook 00: o Executivo Federal enfrenta uma **pressão estrutural de reposição por envelhecimento**, e não por rotatividade voluntária (exoneração responde por apenas ~7%).

Implicação para política pública: o planejamento de concursos deveria ser pautado por projeções demográficas (idade dos servidores ativos), não por modelos de turnover convencionais.


## 3. Normalização e padronização (etapa 6.12)

> *"Você deverá aplicar normalização ou padronização em pelo menos uma variável numérica. Essa etapa é importante quando os dados possuem escalas muito diferentes ou quando podem ser usados em técnicas de Data Mining e Machine Learning."*

### 3.1 Comparativo das três técnicas disponíveis

| Técnica | Fórmula | Faixa de saída | Sensibilidade a outliers |
|---|---|---|---|
| **Min-Max Scaling** | `(x - min) / (max - min)` | [0, 1] | **Alta** — outliers achatam o resto |
| **StandardScaler (Z-score)** | `(x - mean) / std` | aprox. [-3, 3] | **Média** — média e desvio são afetados |
| **RobustScaler** | `(x - mediana) / IQR` | sem limite fixo | **Baixa** — mediana e IQR resistem |

### 3.2 Qual técnica usar nesta base?

Conforme documentado no Notebook 01 (problema P09) e no Notebook 02 (seção 4), a base tem **outliers reais e legítimos** — não erros. A `qtd_aprovada`, por exemplo, vai de 1 (cargos quase extintos) a 34.209 (Técnico Administrativo em Educação). Aplicar MinMax aqui produziria uma distribuição quase totalmente colada em zero, perdendo capacidade de discriminação para 99% dos cargos.

**Decisão técnica:**
- **RobustScaler** será usado como técnica principal — é a única apropriada para a estrutura desta base.
- **MinMax e StandardScaler** serão aplicados em paralelo apenas para fins de **comparação didática** (etapa 6.12 explicitamente pede que se demonstre a diferença "antes e depois").


In [14]:
# Variáveis numéricas-alvo da normalização
cols_normalizar = ['qtd_aprovada', 'qtd_ocupada', 'qtd_vaga', 'total_vacancias', 'deficit_nominal']
print(f"Colunas a normalizar ({len(cols_normalizar)}):")
for c in cols_normalizar:
    print(f"  - {c}")

# Estatísticas ANTES da normalização
print("\n=== ESTATÍSTICAS ANTES DA NORMALIZAÇÃO ===")
df[cols_normalizar].describe().round(2)

Colunas a normalizar (5):
  - qtd_aprovada
  - qtd_ocupada
  - qtd_vaga
  - total_vacancias
  - deficit_nominal

=== ESTATÍSTICAS ANTES DA NORMALIZAÇÃO ===


,qtd_aprovada,qtd_ocupada,qtd_vaga,total_vacancias,deficit_nominal
count,12.769.000,12.769.000,12.769.000,12.769.000,12.769.000
mean,54.850,36.670,18.180,21.940,18.180
std,518.080,262.330,328.600,352.620,328.600
min,1.000,0.000,0.000,0.000,0.000
25%,1.000,1.000,0.000,0.000,0.000
50%,3.000,3.000,0.000,0.000,0.000
75%,12.000,9.000,1.000,2.000,1.000
max,34.209.000,13.302.000,22.911.000,22.787.000,22.911.000


### 3.3 Aplicação do RobustScaler (técnica principal)

In [15]:
scaler_robust = RobustScaler()
dados_robust = scaler_robust.fit_transform(df[cols_normalizar])

# Criar colunas com sufixo _robust
for i, col in enumerate(cols_normalizar):
    df[f'{col}_robust'] = dados_robust[:, i]

print("=== ESTATÍSTICAS APÓS RobustScaler ===")
df[[f'{c}_robust' for c in cols_normalizar]].describe().round(3)

=== ESTATÍSTICAS APÓS RobustScaler ===


,qtd_aprovada_robust,qtd_ocupada_robust,qtd_vaga_robust,total_vacancias_robust,deficit_nominal_robust
count,12.769.000,12.769.000,12.769.000,12.769.000,12.769.000
mean,4.714,4.209,18.181,10.969,18.181
std,47.099,32.791,328.597,176.309,328.597
min,-0.182,-0.375,0.000,0.000,0.000
25%,-0.182,-0.250,0.000,0.000,0.000
50%,0.000,0.000,0.000,0.000,0.000
75%,0.818,0.750,1.000,1.000,1.000
max,3.109.636,1.662.375,22.911.000,11.393.500,22.911.000


**O que essas estatísticas mostram:**

A mediana das colunas `*_robust` é exatamente **0** (por construção do método). Os valores positivos representam cargos acima da mediana; negativos, abaixo. Os máximos continuam altos (como deveriam — outliers legítimos), mas agora estão em escala comparável entre variáveis: um cargo "10× acima da mediana de aprovada" é diretamente comparável a "10× acima da mediana de ocupada".

Essa propriedade torna o RobustScaler ideal para algoritmos de DM sensíveis a escala (K-Means, KNN, PCA) — mantendo a informação dos outliers intacta.


### 3.4 Comparação didática com MinMax e StandardScaler

Aplicamos as outras duas técnicas **apenas em `qtd_aprovada`** para demonstrar visualmente como elas distorceriam os dados desta base. Estas colunas comparativas **não vão para o dataset final** — são geradas só para ilustrar.

In [16]:
scaler_minmax = MinMaxScaler()
scaler_std    = StandardScaler()

qtd_aprovada_minmax = scaler_minmax.fit_transform(df[['qtd_aprovada']]).ravel()
qtd_aprovada_std    = scaler_std.fit_transform(df[['qtd_aprovada']]).ravel()
qtd_aprovada_robust = df['qtd_aprovada_robust'].values

# Comparar a distribuição em percentis
percentis = [0, 1, 25, 50, 75, 99, 100]
print(f"{'Percentil':<12}{'Original':>12}{'MinMax':>12}{'Standard':>12}{'Robust':>12}")
print('-' * 60)
for p in percentis:
    o = np.percentile(df['qtd_aprovada'], p)
    m = np.percentile(qtd_aprovada_minmax, p)
    s = np.percentile(qtd_aprovada_std, p)
    r = np.percentile(qtd_aprovada_robust, p)
    print(f"  p{p:<10}{o:>12.2f}{m:>12.6f}{s:>12.3f}{r:>12.3f}")

Percentil       Original      MinMax    Standard      Robust
------------------------------------------------------------
  p0                 1.00    0.000000      -0.104      -0.182
  p1                 1.00    0.000000      -0.104      -0.182
  p25                1.00    0.000000      -0.104      -0.182
  p50                3.00    0.000058      -0.100       0.000
  p75               12.00    0.000322      -0.083       0.818
  p99             1099.92    0.032125       2.017      99.720
  p100           34209.00    1.000000      65.926    3109.636


**Leitura crítica da tabela:**

- **MinMax**: 75% dos cargos ficam comprimidos abaixo do valor `0.000497` — toda a distribuição fica esmagada no canto inferior do intervalo [0, 1]. A escala perde quase toda a granularidade nos cargos pequenos e médios, que são justamente a maior parte da base.
- **StandardScaler**: a mediana fica em `-0.245` (não em zero), porque a média é puxada para cima pelos outliers. O quantil p99 chega a **+8.7 desvios padrão** — um valor que algoritmos baseados em assumir normalidade interpretariam como anomalia, quando na verdade é só a realidade institucional.
- **RobustScaler**: mediana em zero (por construção). Os outliers continuam visíveis (p99 alto), mas o miolo da distribuição mantém poder de discriminação.

**Conclusão:** o RobustScaler é a escolha tecnicamente correta para a próxima fase analítica. As colunas `qtd_aprovada_robust`, `qtd_ocupada_robust`, `qtd_vaga_robust`, `total_vacancias_robust` e `deficit_nominal_robust` serão preservadas no dataset final.


## 4. Discretização (etapa 6.13)

> *"Você deverá transformar pelo menos uma variável numérica em categorias ou faixas."*

A discretização transforma variáveis contínuas em categorias, permitindo:
- Análises de frequência (frequência de "cargos grandes" por órgão).
- Uso direto em árvores de decisão e regras de associação (DM).
- Comunicação executiva (relatórios falam em "porte do cargo", não em "qtd_aprovada = 35.7").

### 4.1 Duas discretizações com lógicas distintas

| Variável discretizada | Origem | Método | Justificativa |
|---|---|---|---|
| `porte_cargo` | `qtd_aprovada` | `pd.cut` com cortes de negócio | Faixas semanticamente alinhadas à gestão pública |
| `faixa_taxa_ocupacao` | `taxa_ocupacao` | `pd.cut` com cortes operacionais | Status de preenchimento legível por gestor |

### 4.2 Discretização de `qtd_aprovada` → `porte_cargo`

Optamos por **`pd.cut` com cortes definidos por critério de negócio** em vez de `pd.qcut` (quartis). Razão: quartis dividem em grupos com mesmo tamanho, o que não tem significado gerencial. Critérios de negócio produzem categorias **interpretáveis e estáveis** mesmo quando a base muda ao longo do tempo.

In [17]:
# Cortes de negócio para porte do cargo
# Baseados na distribuição observada (Notebook 02b) e em literatura de gestão pública
limites_porte = [-0.01, 2, 10, 50, 500, np.inf]
rotulos_porte = ['Micro (1-2)', 'Pequeno (3-10)', 'Médio (11-50)',
                 'Grande (51-500)', 'Mega (>500)']

df['porte_cargo'] = pd.cut(
    df['qtd_aprovada'],
    bins=limites_porte,
    labels=rotulos_porte,
    include_lowest=True
).astype('category')

# Validação
print("=== DISTRIBUIÇÃO DA VARIÁVEL ORIGINAL vs. DISCRETIZADA ===\n")
print(f"Variável original (qtd_aprovada):")
print(df['qtd_aprovada'].describe().round(2))

print(f"\nVariável discretizada (porte_cargo):")
freq_porte = df['porte_cargo'].value_counts().reindex(rotulos_porte)
pct_porte  = (df['porte_cargo'].value_counts(normalize=True) * 100).reindex(rotulos_porte).round(1)
tabela = pd.DataFrame({'qtd_cargos': freq_porte, 'pct': pct_porte})
print(tabela.to_string())

=== DISTRIBUIÇÃO DA VARIÁVEL ORIGINAL vs. DISCRETIZADA ===

Variável original (qtd_aprovada):
count   12.769.000
mean        54.850
std        518.080
min          1.000
25%          1.000
50%          3.000
75%         12.000
max     34.209.000
Name: qtd_aprovada, dtype: float64

Variável discretizada (porte_cargo):
                 qtd_cargos    pct
porte_cargo                       
Micro (1-2)            5558 43.500
Pequeno (3-10)         3715 29.100
Médio (11-50)          2266 17.700
Grande (51-500)         966  7.600
Mega (>500)             264  2.100


**Leitura da tabela:**

- Cargos **Micro** (1-2 vagas aprovadas) representam quase metade da base — são tipicamente cargos de chefia, comissionados ou em extinção.
- Apenas **2,3%** dos cargos têm porte **Mega** (>500 vagas) — mas concentram a esmagadora maioria dos servidores ativos. É essa pequena fração que aparece como "outliers" no método IQR, conforme já documentado.

Esta discretização é útil para:
- **Relatórios de BI**: "X% do quadro federal está concentrado em cargos de porte Mega".
- **Mining de regras**: associar `porte_cargo` com `nivel`, `em_extincao`, `plano_carreira`.


### 4.3 Discretização de `taxa_ocupacao` → `faixa_taxa_ocupacao`

Cinco faixas com significado operacional direto para o gestor público. Os cortes refletem patamares usados em planejamento de concursos: abaixo de 40% é crítico (justifica concurso emergencial); acima de 100% é excedente (precisa de regularização).

In [18]:
# Cortes operacionais de taxa de ocupação
limites_taxa  = [-0.01, 0.4, 0.7, 0.9, 1.0, np.inf]
rotulos_taxa  = ['Crítica (<40%)', 'Baixa (40-70%)', 'Adequada (70-90%)',
                 'Completa (90-100%)', 'Excedente (>100%)']

df['faixa_taxa_ocupacao'] = pd.cut(
    df['taxa_ocupacao'],
    bins=limites_taxa,
    labels=rotulos_taxa,
    include_lowest=True
).astype('category')

# Validação
print("=== DISTRIBUIÇÃO DA TAXA DE OCUPAÇÃO ===\n")
freq_taxa = df['faixa_taxa_ocupacao'].value_counts().reindex(rotulos_taxa)
pct_taxa  = (df['faixa_taxa_ocupacao'].value_counts(normalize=True) * 100).reindex(rotulos_taxa).round(1)
tabela = pd.DataFrame({'qtd_cargos': freq_taxa, 'pct': pct_taxa})
print(tabela.to_string())

n_nulos = df['faixa_taxa_ocupacao'].isnull().sum()
print(f"\nNulos (cargos sem aprovação legal): {n_nulos:,}".replace(',', '.'))

=== DISTRIBUIÇÃO DA TAXA DE OCUPAÇÃO ===

                     qtd_cargos    pct
faixa_taxa_ocupacao                   
Crítica (<40%)             1539 12.100
Baixa (40-70%)              610  4.800
Adequada (70-90%)           870  6.800
Completa (90-100%)         9750 76.400
Excedente (>100%)             0  0.000

Nulos (cargos sem aprovação legal): 0


**Leitura da tabela:**

A distribuição surpreende: **76,4% dos cargos estão na faixa "Completa" (90-100%)**, e **nenhum cargo** ficou na faixa "Excedente" (>100%). Apenas ~17% dos cargos individuais estão em situação crítica ou baixa.

**Aparente contradição com a agregação por nível (seção 2.2):** lá, a `taxa_ocupacao_agregada` ficou em 62-69% para NS/NI — abaixo do "Adequada". Aqui, a maioria dos cargos individuais aparece como "Completa". Como conciliar?

A explicação é **o paradoxo de Simpson aplicado a proporções**:

- A grande maioria dos cargos individuais (76%) é de **porte Micro/Pequeno** (visto na seção 4.2). Para esses, ter `qtd_ocupada == qtd_aprovada` é trivial — frequentemente `aprovada = ocupada = 1` ou `2`, produzindo `taxa_ocupacao = 100%`.
- Os cargos **Mega** (2% da base) concentram a maior parte do quadro federal — e tipicamente operam com 50-70% de ocupação. Esses arrastam a métrica agregada para baixo.

**Implicação prática:**
- O indicador correto para reportar "saúde do Executivo Federal" é a `taxa_ocupacao_agregada` (≈67%) — não a média das taxas individuais.
- A discretização aqui é útil para **mining individual** (identificar cargos específicos com déficit), não para sumarização do todo.

Cargos na faixa **"Excedente"** seriam casos em que `qtd_ocupada > qtd_aprovada` — situação teoricamente possível por cessões e comissionamentos. Nesta competência específica (03/2026), não há nenhum registro nessa faixa: a regra de ouro `OCUPADA ≤ APROVADA` é respeitada em 100% dos cargos. O problema P08 do diagnóstico (relacionado a `DISTRIBUIDA`, não a `APROVADA`) é capturado pela feature `diferenca_distribuicao`, e não por esta discretização.

## 5. Validação final e exportação

### 5.1 Comparativo antes vs. depois

In [19]:
# Quantas colunas existiam quando carregamos o dataset_pos_limpeza?
COLUNAS_POS_LIMPEZA = 21  # documentado no Notebook 02 (20 originais + is_outlier)
novas = df.shape[1] - COLUNAS_POS_LIMPEZA

print("=== ANTES (dataset_pos_limpeza) ===")
print(f"  Linhas:  12.769")
print(f"  Colunas: {COLUNAS_POS_LIMPEZA}")

print(f"\n=== DEPOIS (dataset com features) ===")
print(f"  Linhas:  {df.shape[0]:,}".replace(',', '.'))
print(f"  Colunas: {df.shape[1]} (+ {novas} novas)")
print(f"  Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n=== NOVAS COLUNAS CRIADAS ===")
print("\nFeature Engineering (etapa 6.14):")
print("  - total_vacancias, pct_aposentadoria, taxa_ocupacao,")
print("    taxa_vacancia_mensal, deficit_nominal, diferenca_distribuicao,")
print("    ano, mes, trimestre")

print("\nNormalização (etapa 6.12):")
print("  - qtd_aprovada_robust, qtd_ocupada_robust, qtd_vaga_robust,")
print("    total_vacancias_robust, deficit_nominal_robust")

print("\nDiscretização (etapa 6.13):")
print("  - porte_cargo, faixa_taxa_ocupacao")

=== ANTES (dataset_pos_limpeza) ===
  Linhas:  12.769
  Colunas: 21

=== DEPOIS (dataset com features) ===
  Linhas:  12.769
  Colunas: 37 (+ 16 novas)


  Memória: 6.40 MB

=== NOVAS COLUNAS CRIADAS ===

Feature Engineering (etapa 6.14):
  - total_vacancias, pct_aposentadoria, taxa_ocupacao,
    taxa_vacancia_mensal, deficit_nominal, diferenca_distribuicao,
    ano, mes, trimestre

Normalização (etapa 6.12):
  - qtd_aprovada_robust, qtd_ocupada_robust, qtd_vaga_robust,
    total_vacancias_robust, deficit_nominal_robust

Discretização (etapa 6.13):
  - porte_cargo, faixa_taxa_ocupacao


In [20]:
# Inspeção final dos tipos
print("=== TIPOS DAS COLUNAS NO DATASET COM FEATURES ===\n")
df.info()

=== TIPOS DAS COLUNAS NO DATASET COM FEATURES ===

<class 'pandas.DataFrame'>
RangeIndex: 12769 entries, 0 to 12768
Data columns (total 37 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   ano_mes                 12769 non-null  datetime64[us]
 1   cod_orgao               12769 non-null  str           
 2   nome_orgao              12769 non-null  str           
 3   sigla_orgao             12769 non-null  str           
 4   nivel                   12769 non-null  category      
 5   cod_cargo               12769 non-null  str           
 6   plano_carreira          12769 non-null  category      
 7   nome_cargo              12769 non-null  str           
 8   qtd_aprovada            12769 non-null  int64         
 9   qtd_distribuida         12769 non-null  int64         
 10  qtd_ocupada             12769 non-null  int64         
 11  qtd_vaga                12769 non-null  int64         
 12  vac_ex

### 5.2 Exportação do dataset intermediário

Salvamos como `dataset_com_features.csv` em `dados_tratados/`. Este é o insumo para o **Notebook 04**, que aplicará a seleção/ordenação final de colunas e gerará o catálogo de dados oficial.

In [21]:
DADOS_TRATADOS.mkdir(parents=True, exist_ok=True)
caminho_saida = DADOS_TRATADOS / 'dataset_com_features.csv'
df.to_csv(caminho_saida, index=False, encoding='utf-8')

# Validação: relê e confere shape
DTYPES_RELER = {
    'cod_orgao': str,
    'cod_cargo': str,
    'em_extincao': 'category',
    'nivel': 'category',
    'plano_carreira': 'category',
    'porte_cargo': 'category',
    'faixa_taxa_ocupacao': 'category',
}
df_check = pd.read_csv(caminho_saida, dtype=DTYPES_RELER, parse_dates=['ano_mes'])

print(f"Arquivo salvo: {caminho_saida}")
print(f"Tamanho: {caminho_saida.stat().st_size / 1024**2:.2f} MB")
print(f"\nReleitura de validação:")
print(f"  Linhas:  {df_check.shape[0]:,}".replace(',', '.'))
print(f"  Colunas: {df_check.shape[1]}")
assert df_check.shape == df.shape, 'Mismatch entre dataframe original e relido'
print('  ✓ Consistência validada.')

Arquivo salvo: /home/claude/PM3_CargosVagosVacancias_v2/dados_tratados/dataset_com_features.csv
Tamanho: 3.49 MB

Releitura de validação:
  Linhas:  12.769
  Colunas: 37
  ✓ Consistência validada.


### 5.3 Exportação das tabelas agregadas

Salvamos também as três agregações da seção 2, para que possam ser consumidas diretamente em ferramentas de BI sem precisar reprocessar o pipeline.

In [22]:
PASTA_AGG = DADOS_TRATADOS / 'agregacoes'
PASTA_AGG.mkdir(parents=True, exist_ok=True)

agg_orgao.to_csv(PASTA_AGG / 'agg_por_orgao.csv', index=False, encoding='utf-8')
agg_nivel.to_csv(PASTA_AGG / 'agg_por_nivel.csv', index=False, encoding='utf-8')
agg_vacancia.to_csv(PASTA_AGG / 'agg_por_tipo_vacancia.csv', encoding='utf-8')

print(f"Agregações exportadas em: {PASTA_AGG}")
for f in sorted(PASTA_AGG.iterdir()):
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

Agregações exportadas em: /home/claude/PM3_CargosVagosVacancias_v2/dados_tratados/agregacoes
  - agg_por_nivel.csv (0.3 KB)
  - agg_por_orgao.csv (10.5 KB)
  - agg_por_tipo_vacancia.csv (0.2 KB)


## 6. Conclusão

### O que foi feito neste notebook

1. **Etapa 6.14 (Feature Engineering):** criamos 9 novas variáveis derivadas — 6 com semântica de gestão de pessoas (`total_vacancias`, `pct_aposentadoria`, `taxa_ocupacao`, `taxa_vacancia_mensal`, `deficit_nominal`, `diferenca_distribuicao`) e 3 temporais (`ano`, `mes`, `trimestre`). Cada uma justificada pela ligação com pelo menos uma das 11 perguntas analíticas do Notebook 00.

2. **Etapa 6.11 (Agregação):** geramos três agregações complementares — por órgão (visão executiva), por nível de escolaridade (comparação P6) e por tipo de vacância (peso relativo das causas). As tabelas foram exportadas em `dados_tratados/agregacoes/` para reuso em BI.

3. **Etapa 6.12 (Normalização):** aplicamos **RobustScaler** em 5 variáveis numéricas como técnica principal, com justificativa técnica explícita (presença de outliers reais e legítimos). Comparamos com MinMax e StandardScaler em uma variável de exemplo para demonstrar o erro que seria adotá-los.

4. **Etapa 6.13 (Discretização):** criamos duas variáveis categóricas — `porte_cargo` (5 faixas semânticas de tamanho institucional) e `faixa_taxa_ocupacao` (5 status operacionais de preenchimento).

### Onde os artefatos foram salvos

| Artefato | Localização |
|---|---|
| Dataset enriquecido | `dados_tratados/dataset_com_features.csv` |
| Agregação por órgão | `dados_tratados/agregacoes/agg_por_orgao.csv` |
| Agregação por nível | `dados_tratados/agregacoes/agg_por_nivel.csv` |
| Agregação por vacância | `dados_tratados/agregacoes/agg_por_tipo_vacancia.csv` |

### Próximo notebook

O **Notebook 04** vai consumir `dataset_com_features.csv`, fazer a seleção final de colunas (eliminando as auxiliares como `is_outlier` se não forem necessárias, ou mantendo-as como flags), reordenar para legibilidade e gerar o **catálogo de dados** documentando cada coluna (origem, tipo, tratamento aplicado, uso esperado) — etapas 6.15 e 6.16 do enunciado.
